In [1]:
import os
import numpy as np
import argparse
import multiprocessing as mp
from multiprocessing import Pool

# Ensure friendly behavior of OpenMP and multiprocessing
os.environ["OMP_NUM_THREADS"] = "1"
mp.set_start_method('spawn', force=True)

import pocomc as pc
from cosmoprimo import Cosmology
from scipy.stats import uniform, norm
from cobaya.model import get_model

# Import likelihood modules (new versions that load data from npz files)
from bao import BAOLikelihood
from cmb import CMBCompressedLikelihood
from supernova import Union3SNLikelihoodSys
from desilike.likelihoods.supernovae import Union3SNLikelihood, PantheonPlusSNLikelihood, DESY5SNLikelihood

In [2]:
cmb_like = None
# if include_cmb:
#     cmb_like = CMBCompressedLikelihood(engine='camb', cosmo=None)
# Define your model and data
info = {
    "theory": {"camb": None},
    "params": {
        # cosmological parameters
        "w": {"prior": {"min": -2, "max": 0}, "ref": -1},
        "wa": {"prior": {"min": -2, "max": 2}, "ref": 0},
        "omch2": {"prior": {"min": 0.05, "max": 0.25}, "ref": 0.1203},
        "ombh2": {"prior": {"min": 0.01, "max": 0.04}, "ref": 0.02218},
        "H0": {"prior": {"min": 40, "max": 100}, "ref": 67.36}, # check if its zeta instead of H0
        "As": {"prior": {"min": 1e-10, "max": 5e-9}, "ref": 2.1e-9, "latex": "A_s"},
        "ns": {"prior": {"min": 0.9, "max": 1.1}, "ref": 0.965},
        "tau": {"prior": {"min": 0.01, "max": 0.8}, "ref": 0.054},

        # nuisance parameters (Planck 2018 high-l TTTEEE) I used cobaya-doc planck_2018_highl_plik.TTTEEE --python
        "A_cib_217": {"prior": {"dist": "uniform", "min": 0, "max": 200}, "ref": 67, "latex": "A^\\mathrm{CIB}_{217}"},
        "A_planck": {"prior": {"dist": "norm", "loc": 1, "scale": 0.0025}, "ref": 1, "latex": "y_\\mathrm{cal}"},
        "A_sz": {"prior": {"dist": "uniform", "min": 0, "max": 10}, "ref": 7, "latex": "A^\\mathrm{tSZ}_{143}"},
        "calib_100T": {"prior": {"dist": "norm", "loc": 1.0002, "scale": 0.0007}, "ref": 1.0002, "latex": "c_{100}"},
        "calib_217T": {"prior": {"dist": "norm", "loc": 0.99805, "scale": 0.00065}, "ref": 0.99805, "latex": "c_{217}"},

        "gal545_A_100": {"prior": {"dist": "norm", "loc": 8.6, "scale": 2}, "ref": 8.6, "latex": "A^\\mathrm{dustTT}_{100}"},
        "gal545_A_143": {"prior": {"dist": "norm", "loc": 10.6, "scale": 2}, "ref": 10.6, "latex": "A^\\mathrm{dustTT}_{143}"},
        "gal545_A_143_217": {"prior": {"dist": "norm", "loc": 23.5, "scale": 8.5}, "ref": 23.5, "latex": "A^\\mathrm{dustTT}_{143\\times217}"},
        "gal545_A_217": {"prior": {"dist": "norm", "loc": 91.9, "scale": 20}, "ref": 91.9, "latex": "A^\\mathrm{dustTT}_{217}"},

        # fixed EE dust parameters
        "galf_EE_A_100": {"value": 0.055, "ref": 0.055, "latex": "A^\\mathrm{dustEE}_{100}"},
        "galf_EE_A_100_143": {"value": 0.04, "ref": 0.04, "latex": "A^\\mathrm{dustEE}_{100\\times143}"},
        "galf_EE_A_100_217": {"value": 0.094, "ref": 0.094, "latex": "A^\\mathrm{dustEE}_{100\\times217}"},
        "galf_EE_A_143": {"value": 0.086, "ref": 0.086, "latex": "A^\\mathrm{dustEE}_{143}"},
        "galf_EE_A_143_217": {"value": 0.21, "ref": 0.21, "latex": "A^\\mathrm{dustEE}_{143\\times217}"},
        "galf_EE_A_217": {"value": 0.7, "ref": 0.7, "latex": "A^\\mathrm{dustEE}_{217}"},

        # TE dust parameters
        "galf_TE_A_100": {"prior": {"dist": "norm", "loc": 0.13, "scale": 0.042}, "ref": 0.13, "latex": "A^\\mathrm{dustTE}_{100}"},
        "galf_TE_A_100_143": {"prior": {"dist": "norm", "loc": 0.13, "scale": 0.036}, "ref": 0.13, "latex": "A^\\mathrm{dustTE}_{100\\times143}"},
        "galf_TE_A_100_217": {"prior": {"dist": "norm", "loc": 0.46, "scale": 0.09}, "ref": 0.46, "latex": "A^\\mathrm{dustTE}_{100\\times217}"},
        "galf_TE_A_143": {"prior": {"dist": "norm", "loc": 0.207, "scale": 0.072}, "ref": 0.207, "latex": "A^\\mathrm{dustTE}_{143}"},
        "galf_TE_A_143_217": {"prior": {"dist": "norm", "loc": 0.69, "scale": 0.09}, "ref": 0.69, "latex": "A^\\mathrm{dustTE}_{143\\times217}"},
        "galf_TE_A_217": {"prior": {"dist": "norm", "loc": 1.938, "scale": 0.54}, "ref": 1.938, "latex": "A^\\mathrm{dustTE}_{217}"},

        # kSZ and point sources
        "ksz_norm": {"prior": {"dist": "uniform", "min": 0, "max": 10}, "ref": 0, "latex": "A^\\mathrm{kSZ}"},
        "ps_A_100_100": {"prior": {"dist": "uniform", "min": 0, "max": 400}, "ref": 257, "latex": "A^\\mathrm{PS}_{100}"},
        "ps_A_143_143": {"prior": {"dist": "uniform", "min": 0, "max": 400}, "ref": 47, "latex": "A^\\mathrm{PS}_{143}"},
        "ps_A_143_217": {"prior": {"dist": "uniform", "min": 0, "max": 400}, "ref": 40, "latex": "A^\\mathrm{PS}_{143\\times217}"},
        "ps_A_217_217": {"prior": {"dist": "uniform", "min": 0, "max": 400}, "ref": 104, "latex": "A^\\mathrm{PS}_{217}"},
        "xi_sz_cib": {"prior": {"dist": "uniform", "min": 0, "max": 1}, "ref": 0, "latex": "\\xi^{\\mathrm{tSZ}\\times\\mathrm{CIB}}"},
    },
    "likelihood": {
        "planck_2018_highl_plik.TTTEEE": None,
        "planck_2018_lowl.TT": None,
        "planck_2018_lowl.EE": None,
        "planck_2018_lensing.clik": None,
    },
    # sampler can be None if I just want likelihoods
    "sampler": None,
    "output": None  # no chains needed
}
# Build the model
cobaya_model = get_model(info)
# cobaya_model.loglikes()
cmb_like = cobaya_model

# List of all parameters with priors (sampled parameters)
cobaya_nuis_names = [name for name, par in info["params"].items() if "prior" in par]
cobaya_info = info["params"]

[prior] *WARNING* External prior 'SZ' loaded. Mind that it might not be normalized!
[camb] `camb` module loaded successfully from /home/eleannakolonia/cosmo_project/cobaya_packages/code/CAMB/camb
[planck_2018_highl_plik.ttteee] `clipy` module loaded successfully from /home/eleannakolonia/cosmo_project/cobaya_packages/code/planck/clipy/clipy
----
clipy_0.15
Checking likelihood '/home/eleannakolonia/cosmo_project/cobaya_packages/data/planck_2018/baseline/plc_3.0/hi_l/plik/plik_rd12_HM_v22b_TTTEEE.clik' on test data. got -1172.47 expected -1172.47 (diff 4.00569e-06)
----
[planck_2018_lensing.clik] `clipy` module loaded successfully from /home/eleannakolonia/cosmo_project/cobaya_packages/code/planck/clipy/clipy
----
clipy_0.15
Checking likelihood '/home/eleannakolonia/cosmo_project/cobaya_packages/data/planck_2018/baseline/plc_3.0/lensing/smicadx12_Dec5_ftl_mv2_ndclpp_p_teb_consext8.clik_lensing' on test data. got -4.42102
----


In [18]:
from cobaya.model import get_model

model = get_model(info)

# Get the sampled parameter names in the order Cobaya expects
sampled_names = model.parameterization.sampled_params()
print("Sampled parameters:", sampled_names)
print(" my array:", allparams)

# Get all derived parameters
derived_names = model.parameterization.derived_params()
print("Derived parameters:", derived_names)

# Optionally, try to get a valid random point and see the full logposterior
point, logpost = model.get_valid_point(max_tries=1, logposterior_as_dict=True)
print("Random valid point:", point)
print("Logposterior dict keys:", logpost.keys())

[prior] *WARNING* External prior 'SZ' loaded. Mind that it might not be normalized!
[camb] `camb` module loaded successfully from /home/eleannakolonia/cosmo_project/cobaya_packages/code/CAMB/camb
[planck_2018_highl_plik.ttteee] `clipy` module loaded successfully from /home/eleannakolonia/cosmo_project/cobaya_packages/code/planck/clipy/clipy
----
clipy_0.15
Checking likelihood '/home/eleannakolonia/cosmo_project/cobaya_packages/data/planck_2018/baseline/plc_3.0/hi_l/plik/plik_rd12_HM_v22b_TTTEEE.clik' on test data. got -1172.47 expected -1172.47 (diff 4.00569e-06)
----
[planck_2018_lensing.clik] `clipy` module loaded successfully from /home/eleannakolonia/cosmo_project/cobaya_packages/code/planck/clipy/clipy
----
clipy_0.15
Checking likelihood '/home/eleannakolonia/cosmo_project/cobaya_packages/data/planck_2018/baseline/plc_3.0/lensing/smicadx12_Dec5_ftl_mv2_ndclpp_p_teb_consext8.clik_lensing' on test data. got -4.42102
----
Sampled parameters: {'w': nan, 'wa': nan, 'omch2': nan, 'ombh2

In [4]:
omega_b =0.022
Omega_m= 0.31
h=0.67
w0=-1
wa=0
As=2.1e-9
ns=0.96
tau=0.06
# Build a full parameter dictionary using Cobaya info
allparams = {}

# Loop over the parameters in the order Cobaya defines them
for key in cobaya_nuis_names:
    param_info = info['params'].get(key)
    if param_info is None:
        raise ValueError(f"Parameter {key} not found in info['params']")
    
    # If the parameter has a 'ref' key, use it; otherwise, use the 'value' key
    allparams[key] = param_info.get('ref', param_info.get('value'))
    
# Check that all parameters are set
print(allparams)
print("Number of parameters:", len(allparams))

{'w': -1, 'wa': 0, 'omch2': 0.1203, 'ombh2': 0.02218, 'H0': 67.36, 'As': 2.1e-09, 'ns': 0.965, 'tau': 0.054, 'A_cib_217': 67, 'A_planck': 1, 'A_sz': 7, 'calib_100T': 1.0002, 'calib_217T': 0.99805, 'gal545_A_100': 8.6, 'gal545_A_143': 10.6, 'gal545_A_143_217': 23.5, 'gal545_A_217': 91.9, 'galf_TE_A_100': 0.13, 'galf_TE_A_100_143': 0.13, 'galf_TE_A_100_217': 0.46, 'galf_TE_A_143': 0.207, 'galf_TE_A_143_217': 0.69, 'galf_TE_A_217': 1.938, 'ksz_norm': 0, 'ps_A_100_100': 257, 'ps_A_143_143': 47, 'ps_A_143_217': 40, 'ps_A_217_217': 104, 'xi_sz_cib': 0}
Number of parameters: 29


In [5]:
allparams.values()

dict_values([-1, 0, 0.1203, 0.02218, 67.36, 2.1e-09, 0.965, 0.054, 67, 1, 7, 1.0002, 0.99805, 8.6, 10.6, 23.5, 91.9, 0.13, 0.13, 0.46, 0.207, 0.69, 1.938, 0, 257, 47, 40, 104, 0])

In [19]:
points=allparams
cmbresults=cmb_like.logposterior(points)
cmbresults.logpost

-1433.2256331746887

In [28]:
params=[0.02218, 0.1203, 67.36, -1, 0, 2.1e-09, 0.965, 0.054, 67, 1, 7, 1.0002, 0.99805, 8.6, 10.6, 23.5, 91.9, 0.13, 0.13, 0.46, 0.207, 0.69, 1.938, 0, 257, 47, 40, 104, 0]
idx = 0
omega_b, omegach2, H0, w0, wa, As, ns, tau = params[idx:idx+8]
idx += 8
# Map PoCoMC params to Cobaya parameter dict
point = {}
# Cosmology
point.update({
    'ombh2': omega_b,
    'omch2': omegach2,
    'H0': H0,
    'w': w0,
    'wa': wa,
    # Use reference values for As, ns, tau unless you sample them explicitly
    'As': As,
    'ns': ns,
    'tau': tau,
})

print('point before nuisance:',point)
print("index", idx)
# Nuisance parameters
for name in cobaya_nuis_names[8:]:
    if idx >= len(params):
        print("error", idx, len(params))
    # return -np.inf
    else:
        point[name] = params[idx]
        idx += 1
        print("ok", idx)

print(point)
try:
    lp = cmb_like.logposterior(point)        # lp is a dict
    ll_cmb = lp.logpost                     # sum of all Planck contributions
    print(logpost)
    # if not np.isfinite(ll_cmb):
    #     return -np.inf
    # total_ll += ll_cmb
except Exception as e:
    print("Cobaya evaluation failed for point:", point)
    print("Exception:", e)
    # return -np.inf

point before nuisance: {'ombh2': 0.02218, 'omch2': 0.1203, 'H0': 67.36, 'w': -1, 'wa': 0, 'As': 2.1e-09, 'ns': 0.965, 'tau': 0.054}
index 8
ok 9
ok 10
ok 11
ok 12
ok 13
ok 14
ok 15
ok 16
ok 17
ok 18
ok 19
ok 20
ok 21
ok 22
ok 23
ok 24
ok 25
ok 26
ok 27
ok 28
ok 29
{'ombh2': 0.02218, 'omch2': 0.1203, 'H0': 67.36, 'w': -1, 'wa': 0, 'As': 2.1e-09, 'ns': 0.965, 'tau': 0.054, 'A_cib_217': 67, 'A_planck': 1, 'A_sz': 7, 'calib_100T': 1.0002, 'calib_217T': 0.99805, 'gal545_A_100': 8.6, 'gal545_A_143': 10.6, 'gal545_A_143_217': 23.5, 'gal545_A_217': 91.9, 'galf_TE_A_100': 0.13, 'galf_TE_A_100_143': 0.13, 'galf_TE_A_100_217': 0.46, 'galf_TE_A_143': 0.207, 'galf_TE_A_143_217': 0.69, 'galf_TE_A_217': 1.938, 'ksz_norm': 0, 'ps_A_100_100': 257, 'ps_A_143_143': 47, 'ps_A_143_217': 40, 'ps_A_217_217': 104, 'xi_sz_cib': 0}
{'logpost': -1433.2256331746887, 'logpriors': {'0': 2.735682595860075, 'SZ': -2.1781063774283376}, 'loglikes': {'planck_2018_highl_plik.TTTEEE': -1219.49742260034, 'planck_2018_lowl.